Designing Nutrition rules

Let's think like a nutrionist

| Health Condition | Calories | Protein | Carbs | Fat | Fiber | Max Sugar | Max Sodium |
| ---------------- | -------: | ------: | ----: | --: | ----: | --------: | ---------: |
| Healthy          |     2200 |      75 |   275 |  70 |    30 |        50 |       2300 |
| Underweight      |     2600 |      90 |   330 |  80 |    30 |        60 |       2300 |
| Diabetes         |     1800 |      90 |   180 |  55 |    35 |        25 |       1800 |
| Hypertension     |     2000 |      80 |   240 |  60 |    35 |        40 |       1500 |
| Obesity          |     1700 |     100 |   160 |  45 |    40 |        20 |       1800 |
| Heart Disease    |     1900 |      85 |   220 |  50 |    35 |        30 |       1500 |

These are not exact medical prescriptions. They are simplified, educational targets suitable for a machine learning.

ML model will predict exactly these.

Target Calories

Target Protein

Target Carbs

Target Fat

Target Fiber

Max Sugar

Max Sodium

Later,

recommendation engine searches

food_nutrition.csv

to find foods matching those targets.

In [1]:
import pandas as pd
import numpy as np

Load Patient Dataset

In [2]:
patient_df = pd.read_csv("../data/processed/patient_profiles.csv")

In [3]:
patient_df.head()

,age,gender,height_m,weight_kg,bmi,health_condition,activity_level,diet_type,allergy
0,71,Female,1.61,112,26.5,Diabetes,Low,Non-Vegetarian,NaN
1,56,Female,1.79,111,19.1,Healthy,Medium,Non-Vegetarian,NaN
2,34,Female,1.72,96,19.3,Healthy,Low,Vegan,Dairy
3,79,Female,1.68,114,25.9,Healthy,Medium,Vegetarian,Dairy
4,67,Male,1.65,47,19.8,Hypertension,Low,Non-Vegetarian,Seafood


this notebook's job is to add target columns.

In [4]:
nutrition_rules = pd.DataFrame({

    "health_condition":[
        "Healthy",
        "Underweight",
        "Diabetes",
        "Hypertension",
        "Obesity",
        "Heart Disease"
    ],

    "base_calories":[
        2200,
        2600,
        1800,
        2000,
        1700,
        1900
    ],

    "protein":[
        75,
        90,
        90,
        80,
        100,
        85
    ],

    "carbohydrates":[
        275,
        330,
        180,
        240,
        160,
        220
    ],

    "fat":[
        70,
        80,
        55,
        60,
        45,
        50
    ],

    "fiber":[
        30,
        30,
        35,
        35,
        40,
        35
    ],

    "max_sugar":[
        50,
        60,
        25,
        40,
        20,
        30
    ],

    "max_sodium":[
        2300,
        2300,
        1800,
        1500,
        1800,
        1500
    ]

})

these are base values, not final targets.

We will manipulate them according to patient's condition.

In [5]:
nutrition_rules

,health_condition,base_calories,protein,carbohydrates,fat,fiber,max_sugar,max_sodium
0,Healthy,2200,75,275,70,30,50,2300
1,Underweight,2600,90,330,80,30,60,2300
2,Diabetes,1800,90,180,55,35,25,1800
3,Hypertension,2000,80,240,60,35,40,1500
4,Obesity,1700,100,160,45,40,20,1800
5,Heart Disease,1900,85,220,50,35,30,1500


In [6]:
nutrition_rules.to_csv(
    "../data/processed/nutrition_rules.csv",
    index=False
)

Merge Patient Data with Rules

In [7]:
training_df = patient_df.merge(
    nutrition_rules,
    on="health_condition",
    how="left"
)

In [8]:
training_df.head()

,age,gender,height_m,weight_kg,bmi,health_condition,activity_level,diet_type,allergy,base_calories,protein,carbohydrates,fat,fiber,max_sugar,max_sodium
0,71,Female,1.61,112,26.5,Diabetes,Low,Non-Vegetarian,NaN,1800,90,180,55,35,25,1800
1,56,Female,1.79,111,19.1,Healthy,Medium,Non-Vegetarian,NaN,2200,75,275,70,30,50,2300
2,34,Female,1.72,96,19.3,Healthy,Low,Vegan,Dairy,2200,75,275,70,30,50,2300
3,79,Female,1.68,114,25.9,Healthy,Medium,Vegetarian,Dairy,2200,75,275,70,30,50,2300
4,67,Male,1.65,47,19.8,Hypertension,Low,Non-Vegetarian,Seafood,2000,80,240,60,35,40,1500


Dynamic Nutrition Adjustment

calories naturally vary with:

Age

BMI

Activity

Protein, fiber, sodium, etc., are mostly determined by the health condition. We can refine them later if needed.

In [9]:
def adjust_calories(row):

    calories = row["base_calories"]

    # Activity Level
    if row["activity_level"] == "High":
        calories += 300

    elif row["activity_level"] == "Low":
        calories -= 200

    # Age
    if row["age"] <= 30:
        calories += 100

    elif row["age"] > 60:
        calories -= 150

    # BMI
    if row["bmi"] < 18.5:
        calories += 250

    elif row["bmi"] >= 30:
        calories -= 200

    elif row["bmi"] >= 25:
        calories -= 100

    return calories

create the personalized calorie target.

In [10]:
training_df["target_calories"] = training_df.apply(
    adjust_calories,
    axis=1
)

In [11]:
training_df[
    [
        "health_condition",
        "age",
        "bmi",
        "activity_level",
        "base_calories",
        "target_calories"
    ]
].head(20)

,health_condition,age,bmi,activity_level,base_calories,target_calories
0,Diabetes,71,26.5,Low,1800,1350
1,Healthy,56,19.1,Medium,2200,2200
2,Healthy,34,19.3,Low,2200,2000
3,Healthy,79,25.9,Medium,2200,1950
4,Hypertension,67,19.8,Low,2000,1650
5,Obesity,35,26.2,Low,1700,1400
6,Obesity,28,32.3,High,1700,1900
7,Healthy,19,21.4,Medium,2200,2300
8,Hypertension,32,27.5,High,2000,2200
9,Healthy,55,20.5,Medium,2200,2200


In [12]:
training_df["target_calories"].describe()

count    20000.000000
mean      1934.667500
std        367.677235
min       1150.000000
25%       1700.000000
50%       2000.000000
75%       2200.000000
max       3250.000000
Name: target_calories, dtype: float64

In [13]:
training_df[
    training_df["activity_level"] == "High"
][[
    "age",
    "bmi",
    "health_condition",
    "target_calories"
]].head(10)

,age,bmi,health_condition,target_calories
6,28,32.3,Obesity,1900
8,32,27.5,Hypertension,2200
17,19,33.2,Diabetes,2000
19,48,23.7,Healthy,2500
20,45,19.2,Healthy,2500
25,77,21.8,Healthy,2350
28,62,23.8,Healthy,2350
31,36,23.9,Diabetes,2100
35,18,33.8,Diabetes,2000
40,48,21.7,Diabetes,2100


In [14]:
training_df[
    training_df["bmi"] >= 30
][[
    "bmi",
    "target_calories"
]].head(10)

,bmi,target_calories
6,32.3,1900
15,33.9,1800
17,33.2,2000
35,33.8,2000
38,34.2,1300
45,38.5,1600
49,32.7,1450
55,34.8,1800
58,34.0,1150
64,37.3,1350


 adjust_nutrition_targets()

 It will: Take one patient (a row from the DataFrame). 
 Read the base nutrition values. 
 Apply adjustments to calories, protein, carbohydrates, fat, and fiber. 
 Leave sugar and sodium mostly unchanged (since they're primarily disease-specific). 
 Return all adjusted targets together.

In [15]:
def adjust_nutrition_targets(row):

    # ----------------------------
    # Read base values
    # ----------------------------
    calories = row["base_calories"]
    protein = row["protein"]
    carbs = row["carbohydrates"]
    fat = row["fat"]
    fiber = row["fiber"]

    max_sugar = row["max_sugar"]
    max_sodium = row["max_sodium"]

    # ============================
    # Activity Adjustments
    # ============================

    if row["activity_level"] == "High":
        calories += 300
        protein += 10
        carbs += 20
        fat += 5

    elif row["activity_level"] == "Low":
        calories -= 200
        carbs -= 15

    # ============================
    # Age Adjustments
    # ============================

    if row["age"] <= 30:
        calories += 100

    elif row["age"] > 60:
        calories -= 150

    # ============================
    # BMI Adjustments
    # ============================

    if row["bmi"] < 18.5:
        calories += 250
        protein += 15
        carbs += 20

    elif row["bmi"] >= 30:
        calories -= 200
        protein += 5
        fat -= 5
        fiber += 5

    elif row["bmi"] >= 25:
        calories -= 100
        carbs -= 10

    # ============================
    # Safety Limits
    # ============================

    calories = max(1400, min(calories, 3200))

    protein = max(40, protein)
    carbs = max(100, carbs)
    fat = max(30, fat)
    fiber = max(20, fiber)

    # ============================
    # Return Everything
    # ============================

    return pd.Series({
        "target_calories": calories,
        "target_protein": protein,
        "target_carbohydrates": carbs,
        "target_fat": fat,
        "target_fiber": fiber,
        "target_max_sugar": max_sugar,
        "target_max_sodium": max_sodium
    })

In [16]:
nutrition_targets = training_df.apply(
    adjust_nutrition_targets,
    axis=1
)

Join the New Columns

In [17]:
training_df = pd.concat(
    [training_df, nutrition_targets],
    axis=1
)

In [18]:
training_df[
    [
        "health_condition",
        "activity_level",
        "bmi",
        "target_calories",
        "target_protein",
        "target_carbohydrates",
        "target_fat",
        "target_fiber",
        "target_max_sugar",
        "target_max_sodium"
    ]
].head(10)

,health_condition,activity_level,bmi,target_calories,target_calories,target_protein,target_carbohydrates,target_fat,target_fiber,target_max_sugar,target_max_sodium
0,Diabetes,Low,26.5,1350,1400,90,155,55,35,25,1800
1,Healthy,Medium,19.1,2200,2200,75,275,70,30,50,2300
2,Healthy,Low,19.3,2000,2000,75,260,70,30,50,2300
3,Healthy,Medium,25.9,1950,1950,75,265,70,30,50,2300
4,Hypertension,Low,19.8,1650,1650,80,225,60,35,40,1500
5,Obesity,Low,26.2,1400,1400,100,135,45,40,20,1800
6,Obesity,High,32.3,1900,1900,115,180,45,45,20,1800
7,Healthy,Medium,21.4,2300,2300,75,275,70,30,50,2300
8,Hypertension,High,27.5,2200,2200,90,250,65,35,40,1500
9,Healthy,Medium,20.5,2200,2200,75,275,70,30,50,2300


In [19]:
training_df[
    [
        "target_calories",
        "target_protein",
        "target_carbohydrates",
        "target_fat",
        "target_fiber"
    ]
].describe()

,target_calories,target_calories,target_protein,target_carbohydrates,target_fat,target_fiber
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000,20000.000000
mean,1934.667500,1944.340000,86.278000,233.322000,61.924750,34.135750
std,367.677235,350.699464,11.638664,52.076385,11.439542,5.225245
min,1150.000000,1400.000000,75.000000,135.000000,40.000000,30.000000
25%,1700.000000,1700.000000,75.000000,180.000000,55.000000,30.000000
50%,2000.000000,2000.000000,85.000000,260.000000,70.000000,30.000000
75%,2200.000000,2200.000000,95.000000,275.000000,70.000000,40.000000
max,3250.000000,3200.000000,115.000000,370.000000,85.000000,45.000000


This duplicate target_calorie column is created because

training_df["target_calories"] ==> we created this first

and then later adjust_nutrition_targets() function returned another target_calories.

So now the DataFrame has two columns with the same name.

remove the old column

In [20]:
training_df = training_df.drop(columns=["target_calories"])

In [21]:
nutrition_targets = training_df.apply(
    adjust_nutrition_targets,
    axis=1
)

training_df = pd.concat(
    [training_df, nutrition_targets],
    axis=1
)

The model should never see the base rules.

Otherwise it will cheat.

That's data leakage.

Remove Rule Columns like base_calories.

In [22]:
training_df = training_df.drop(columns=[
    "base_calories",
    "protein",
    "carbohydrates",
    "fat",
    "fiber",
    "max_sugar",
    "max_sodium"
])

In [23]:
training_df.columns

Index(['age', 'gender', 'height_m', 'weight_kg', 'bmi', 'health_condition',
       'activity_level', 'diet_type', 'allergy', 'target_protein',
       'target_carbohydrates', 'target_fat', 'target_fiber',
       'target_max_sugar', 'target_max_sodium', 'target_calories',
       'target_protein', 'target_carbohydrates', 'target_fat', 'target_fiber',
       'target_max_sugar', 'target_max_sodium'],
      dtype='str')

Let's save ML training dataset

In [24]:
training_df.to_csv(
    "../data/processed/training_dataset.csv",
    index=False
)